# CNN baseline full pipeline on Google Colab (beginner)

Runs `scripts/cnn_baseline_full_pipeline.py`: **four arms** (PCam raw / PCam Macenko preprocessed / CAMELYON17 raw / CAMELYON17 Macenko preprocessed), then **in-domain and cross-domain evaluation** with the same reporting rules as in the methodology.

This notebook follows the **same layout** as `colab_train_virchow_wilds_preprocessed_beginner.ipynb`: Drive for **outputs** and source data, optional **copy to Colab SSD** for faster HDF5 reads, then a short **artefact check** cell.

**What the script expects under the repo** (after the data cell runs): `pcam_data/training/`, `pcam_data/val/`, and `pcam_data/test/` each containing the raw split `.h5` pairs, plus `pcam_data/preprocessed_macenko_benchmark_style/` for PCam Macenko. CAM17 must live under **`data/wilds/camelyon17_h5_full_oodval/`** with raw split `.h5` files at that level and **`preprocessed_macenko_benchmark_style/`** inside it for Macenko CAM17. **Your Drive layout** can stay under `GP_ECG_DATA/`: `pcam_raw/` whose **training / val / test** subfolders hold the raw PCam `.h5` files, a sibling `preprocessed_macenko_benchmark_style/` for PCam Macenko, **`wilds_raw/`** for raw CAM17 `.h5` files, and **`wilds/preprocessed_macenko_benchmark_style/`** for CAM17 Macenko. The notebook **copies** `pcam_raw` into `pcam_data` (preserving those three folders) and **merges** `wilds_raw` + `wilds/preprocessed_...` into `data/wilds/camelyon17_h5_full_oodval/` on the Colab SSD.

Script defaults: **10 epochs** per arm, batch **320** train and eval, **`num_workers=0`**, progress lines at **10%** batches (no tqdm). Set **`GP_ECG_ROOT`** to the repo. Use **`--resume`** with the same **`--run-id`** after a disconnect.

**Drive copy errors:** If you see `Transport endpoint is not connected` (errno **107**) while copying from Drive, that is the Colab **Drive FUSE** dropping mid-read. The data cell retries a few times and deletes a **partial** local folder; you can increase **`COPY_RETRIES`** / **`COPY_RETRY_SLEEP_SEC`** in the config cell, rerun **`drive.mount(..., force_remount=True)`**, or set **`FORCE_REFRESH_LOCAL_DATA = True`** once to wipe `/content/local_gp_ecg_data` and recopy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
import os
%cd /content
if not os.path.isdir('/content/GP_ECG'):
    !git clone https://github.com/LeenRayess/GP_ECG.git GP_ECG
else:
    print('Repo already present: /content/GP_ECG')
%cd /content/GP_ECG

If you already uploaded the repo into Drive instead of cloning, **skip** the clone cell above and set `REPO_DIR` in the next cell to that path (and `%cd` there once before installs).

In [ ]:
# ==== EDIT THESE LINES ====
REPO_DIR = '/content/GP_ECG'

# Parent folder on Google Drive (e.g. MyDrive/GP_ECG_DATA) — contains pcam_raw, preprocessed_..., wilds, etc.
DRIVE_GP_ECG_DATA = '/content/drive/MyDrive/GP_ECG_DATA'

from pathlib import Path

_dp = Path(DRIVE_GP_ECG_DATA)
# PCam on Drive: pcam_raw must contain training/, val/, test/ (H5 files inside those folders, not loose in pcam_raw).
DRIVE_PCAM_RAW = str(_dp / 'pcam_raw')
DRIVE_PCAM_PREPROCESSED = str(_dp / 'preprocessed_macenko_benchmark_style')
# CAM17 on Drive: raw .h5 pack + preprocessed (merged into camelyon17_h5_full_oodval on SSD).
DRIVE_CAM17_RAW = str(_dp / 'wilds_raw')
DRIVE_CAM17_PRE = str(_dp / 'wilds' / 'preprocessed_macenko_benchmark_style')

# True -> single Drive folder GP_ECG_DATA/pcam_data with training/val/test AND preprocessed inside it (old layout).
USE_SINGLE_DRIVE_PCAM_FOLDER = False

# All pipeline outputs (run_manifest, per-arm checkpoints, evaluation/)
OUT_ROOT = '/content/drive/MyDrive/GP_ECG_RUNS/cnn_baseline_full'

RUN_ID = None  # e.g. 'run_20260212_153000' to resume; None starts a new timestamped folder
EPOCHS = 10  # same default length as train_virchow_preprocessed_colab
RESUME = False
SKIP_TRAIN = False
SKIP_EVAL = False

# Copy large H5 trees from Drive -> Colab SSD once per runtime (recommended).
COPY_DATA_TO_LOCAL_SSD = True

# Google Drive FUSE often throws errno 107 mid-copy; retries + delete partial folder help.
COPY_RETRIES = 5
COPY_RETRY_SLEEP_SEC = 20
# True = always delete /content/local_gp_ecg_data/{pcam_data,data} before copying (slow but clean).
FORCE_REFRESH_LOCAL_DATA = False

import os

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
os.environ['GP_ECG_ROOT'] = REPO_DIR

print('REPO_DIR:', REPO_DIR)
print('DRIVE_GP_ECG_DATA:', DRIVE_GP_ECG_DATA)
print('OUT_ROOT:', OUT_ROOT)
print('GP_ECG_ROOT:', os.environ['GP_ECG_ROOT'])
print('Drive PCam raw:', DRIVE_PCAM_RAW)
print('Drive PCam preprocessed:', DRIVE_PCAM_PREPROCESSED)
print('Drive CAM17 raw (wilds_raw):', DRIVE_CAM17_RAW)
print('Drive CAM17 preprocessed:', DRIVE_CAM17_PRE)

In [ ]:
import errno
import os
import shutil
import time
from pathlib import Path

REPO = Path(REPO_DIR)
LOCAL = Path("/content/local_gp_ecg_data")
dst_pcam = LOCAL / "pcam_data"
dst_wilds = LOCAL / "data" / "wilds"
dst_cam17 = dst_wilds / "camelyon17_h5_full_oodval"

src_pcam_raw = Path(DRIVE_PCAM_RAW)
src_pcam_pre = Path(DRIVE_PCAM_PREPROCESSED)
src_cam17_raw = Path(DRIVE_CAM17_RAW)
src_cam17_pre = Path(DRIVE_CAM17_PRE)
src_single_pcam = Path(DRIVE_GP_ECG_DATA) / "pcam_data"

RETRIES = int(globals().get("COPY_RETRIES", 5))
SLEEP = int(globals().get("COPY_RETRY_SLEEP_SEC", 20))
FORCE = bool(globals().get("FORCE_REFRESH_LOCAL_DATA", False))


def _link_replace(link: Path, target: Path) -> None:
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.is_dir():
        raise RuntimeError(
            f"{link} exists as a real directory; remove or rename it so we can symlink to SSD data."
        )
    link.parent.mkdir(parents=True, exist_ok=True)
    link.symlink_to(target, target_is_directory=True)


def _rm_if_exists(p: Path) -> None:
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)


def _copytree_retry(src: Path, dst: Path, label: str) -> None:
    # shutil.copytree with retries for Google Drive FUSE (errno 107 ENOTCONN, I/O glitches).
    for attempt in range(1, RETRIES + 1):
        try:
            if dst.exists():
                _rm_if_exists(dst)
            print(f"[{label}] copy attempt {attempt}/{RETRIES}: {src} -> {dst}")
            shutil.copytree(src, dst)
            return
        except OSError as e:
            err = getattr(e, "errno", None)
            msg = str(e).lower()
            transient = err in (107, errno.EIO, errno.ENOTCONN, errno.EBUSY) or "transport endpoint" in msg
            if dst.exists():
                print(f"  removing partial destination after error: {e!r}")
                try:
                    _rm_if_exists(dst)
                except OSError:
                    pass
            if attempt >= RETRIES or not transient:
                print("Tip: remount Drive (rerun mount cell with force_remount=True), wait for sync, or set FORCE_REFRESH_LOCAL_DATA=True.")
                raise
            print(f"  Drive I/O glitch ({e!r}); sleeping {SLEEP}s then retry...")
            time.sleep(SLEEP)


def _build_cam17_pack() -> None:
    if not src_cam17_raw.is_dir():
        raise FileNotFoundError("CAM17 raw H5 folder (Drive, e.g. wilds_raw):\n  " + str(src_cam17_raw))
    if not src_cam17_pre.is_dir():
        raise FileNotFoundError("CAM17 Macenko folder (Drive, e.g. wilds/preprocessed_...):\n  " + str(src_cam17_pre))
    marker = dst_cam17 / "train_x.h5"
    if marker.is_file():
        print("Using existing local CAM17 pack:", dst_cam17)
        return
    print("Building data/wilds/camelyon17_h5_full_oodval from Drive (wilds_raw + wilds/preprocessed)...")
    dst_wilds.mkdir(parents=True, exist_ok=True)
    if dst_cam17.exists():
        _rm_if_exists(dst_cam17)
    _copytree_retry(src_cam17_raw, dst_cam17, "cam17_raw")
    dst_pre = dst_cam17 / "preprocessed_macenko_benchmark_style"
    if dst_pre.exists():
        _rm_if_exists(dst_pre)
    _copytree_retry(src_cam17_pre, dst_pre, "cam17_preprocessed")
    print("  done:", dst_cam17)


if not COPY_DATA_TO_LOCAL_SSD:
    raise RuntimeError(
        "COPY_DATA_TO_LOCAL_SSD must be True so wilds_raw + wilds/preprocessed can merge into "
        "data/wilds/camelyon17_h5_full_oodval/."
    )

if FORCE:
    print("FORCE_REFRESH_LOCAL_DATA: clearing local cache...")
    _rm_if_exists(dst_pcam)
    _rm_if_exists(dst_wilds)

if USE_SINGLE_DRIVE_PCAM_FOLDER:
    if not src_single_pcam.is_dir():
        raise FileNotFoundError("Expected folder:\n  " + str(src_single_pcam))
    marker = dst_pcam / "training" / "camelyonpatch_level_2_split_train_x.h5"
    if dst_pcam.is_dir() and not marker.is_file():
        print("Removing incomplete local pcam_data...")
        _rm_if_exists(dst_pcam)
    if not dst_pcam.is_dir():
        _copytree_retry(src_single_pcam, dst_pcam, "pcam_data_single")
        print("  done:", dst_pcam)
    else:
        print("Using existing local pcam_data:", dst_pcam)
else:
    if not src_pcam_raw.is_dir():
        raise FileNotFoundError("Expected PCam raw splits (training/val/test inside):\n  " + str(src_pcam_raw))
    if not src_pcam_pre.is_dir():
        raise FileNotFoundError("Expected PCam Macenko folder:\n  " + str(src_pcam_pre))
    marker = dst_pcam / "training" / "camelyonpatch_level_2_split_train_x.h5"
    if dst_pcam.is_dir() and not marker.is_file():
        print("Removing incomplete local pcam_data from a failed Drive copy...")
        _rm_if_exists(dst_pcam)
    if not dst_pcam.is_dir():
        print("Merging Drive -> local pcam_data/ (pcam_raw + preprocessed_macenko_benchmark_style)...")
        _copytree_retry(src_pcam_raw, dst_pcam, "pcam_raw")
        dst_pre = dst_pcam / "preprocessed_macenko_benchmark_style"
        if dst_pre.exists():
            print("  already present:", dst_pre)
        else:
            print("  copying PCam preprocessed ->", dst_pre)
            _copytree_retry(src_pcam_pre, dst_pre, "pcam_preprocessed")
        print("  done:", dst_pcam)
    else:
        print("Using existing local pcam_data:", dst_pcam)

_build_cam17_pack()

_link_replace(REPO / "pcam_data", dst_pcam)
_link_replace(REPO / "data" / "wilds", dst_wilds)
print("Symlinks OK:")
print(" ", REPO / "pcam_data", "->", dst_pcam.resolve())
print(" ", REPO / "data" / "wilds", "->", dst_wilds.resolve())


In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install h5py scikit-learn numpy

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
from pathlib import Path

REPO = Path(REPO_DIR)
checks = [
    REPO / 'pcam_data' / 'training' / 'camelyonpatch_level_2_split_train_x.h5',
    REPO / 'pcam_data' / 'preprocessed_macenko_benchmark_style' / 'train_x.h5',
    REPO / 'data' / 'wilds' / 'camelyon17_h5_full_oodval' / 'train_x.h5',
    REPO / 'data' / 'wilds' / 'camelyon17_h5_full_oodval' / 'preprocessed_macenko_benchmark_style' / 'train_x.h5',
]
for p in checks:
    print(('OK ' if p.is_file() else 'MISSING '), p)

Run the full pipeline below. Notes:
- Add **`--resume`** after a disconnect; reuse the same **`RUN_ID`** printed at the start of the run (or set `RUN_ID` in the config cell before re-running).
- **`SKIP_TRAIN`** / **`SKIP_EVAL`** only if you already have checkpoints or only want eval.
- Outputs live under **`OUT_ROOT`** on Drive (`<run_id>/`, `evaluation/metrics_all.json`).

In [ ]:
%cd {REPO_DIR}

_extra = []
if RUN_ID:
    _extra += ['--run-id', str(RUN_ID)]
if RESUME:
    _extra.append('--resume')
if SKIP_TRAIN:
    _extra.append('--skip-train')
if SKIP_EVAL:
    _extra.append('--skip-eval')

import os
import shlex
import subprocess

cmd = [
    'python', '-u', 'scripts/cnn_baseline_full_pipeline.py',
    '--epochs', str(EPOCHS),
    '--batch-size', '320',
    '--eval-batch-size', '320',
    '--repo-root', REPO_DIR,
    '--out-root', OUT_ROOT,
] + _extra
print('Running:', ' '.join(shlex.quote(c) for c in cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['GP_ECG_ROOT'] = REPO_DIR
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

In [ ]:
import json
import os
from pathlib import Path

out = Path(OUT_ROOT)
if RUN_ID:
    run_dir = out / str(RUN_ID)
else:
    cands = sorted([p for p in out.iterdir() if p.is_dir() and p.name.startswith('run_')])
    run_dir = cands[-1] if cands else None

if run_dir is None or not run_dir.is_dir():
    print('No run folder found under OUT_ROOT yet:', out)
else:
    print('Run dir:', run_dir)
    manifest = run_dir / 'run_manifest.json'
    metrics = run_dir / 'evaluation' / 'metrics_all.json'
    for label, p in [('run_manifest', manifest), ('metrics_all', metrics)]:
        print(('OK ' if p.is_file() else 'MISSING '), label, p)
    arms = ['pcam_raw', 'pcam_preprocessed', 'cam17_raw', 'cam17_preprocessed']
    for arm in arms:
        ck = run_dir / arm / 'last_checkpoint.pt'
        print(('OK ' if ck.is_file() else 'MISSING '), arm, 'last_checkpoint.pt')